In [1]:
import pandas as pd
import glob
import os

# Mapping dictionary for clean display names
name_dictionary = {
    "hydroelectric_pumped_storage": "Pumped Hydro", 
    "nuclear": "Nuclear", 
    "natural_gas_fired_combined_cycle": "Combined Cycle NG", 
    "biomass": "Biomass", 
    "natural_gas_fired_combustion_turbine": "Combustion Turbine NG", 
    "conventional_steam_coal": "Coal", 
    "batteries": "Battery Storage", 
    "conventional_hydroelectric": "Hydroelectric", 
    "solar_photovoltaic": "Solar", 
    "natural_gas_steam_turbine": "Steam Turbine NG", 
    "offshore_wind_turbine": "Offshore Wind", 
    "distributed_generation": "Distributed Generation",
    "onshore_wind_turbine": "Onshore Wind", 
    "petroleum_liquids": "Petroleum"
}

# Function to extract generator type from column name 
def get_generator_type(col_name):
    first_idx = col_name.find('_')
    last_idx = col_name.rfind('_')
    if first_idx != -1 and last_idx != -1 and first_idx != last_idx:
        return col_name[first_idx+1:last_idx]
    return col_name

# Use glob to find all results folders
folder_pattern = "Scenarios/Run_Variability_*_*_R*/results"
result_folders = glob.glob(folder_pattern)

if not result_folders:
    print("No folders found. Please check your working directory and folder path.")
else:
    print(f"Found {len(result_folders)} scenario folders. Processing...")

    # Master dictionary to store accreditation percentages across all scenarios
    master_accreditation = {}

    for folder in result_folders:
        nse_path = os.path.join(folder, "nse.csv")
        power_path = os.path.join(folder, "power.csv")
        cap_path = os.path.join(folder, "capacity.csv")
        
        if not (os.path.exists(nse_path) and os.path.exists(power_path) and os.path.exists(cap_path)):
            continue
            
        # 1. Read NSE and identify hours > 1 GW (1000 MW)
        nse_df = pd.read_csv(nse_path, index_col=0)
        nse_df = nse_df.drop(['Zone', 'AnnualSum'], errors='ignore')
        
        # Calculate total hourly NSE across all zones
        hourly_nse = nse_df.apply(pd.to_numeric, errors='coerce').sum(axis=1)
        
        # NEW METHODOLOGY: Filter for hours where total NSE > 100
        relevant_hours = hourly_nse[hourly_nse > 100].index.tolist()
        
        if not relevant_hours:
            continue

        # 2. Read Capacity
        cap_df = pd.read_csv(cap_path)
        cap_df['GenType'] = cap_df['Resource'].apply(get_generator_type)
        total_cap_by_type = cap_df.groupby('GenType')['EndCap'].sum().to_dict()
        
        for gen_type in total_cap_by_type.keys():
            if gen_type not in master_accreditation:
                master_accreditation[gen_type] = []
                
        # 3. Read Power and calculate performance for relevant hours
        power_df = pd.read_csv(power_path, index_col=0)
        
        for hr in relevant_hours:
            if hr not in power_df.index:
                continue
            
            row = power_df.loc[hr].drop('Total', errors='ignore')
            
            type_sums = {}
            for col_name, val in row.items():
                gen_type = get_generator_type(col_name)
                type_sums[gen_type] = type_sums.get(gen_type, 0) + float(val)
                
            for gen_type in total_cap_by_type.keys():
                gen_val = type_sums.get(gen_type, 0)
                cap_val = total_cap_by_type.get(gen_type, 0)
                
                if cap_val > 0:
                    master_accreditation[gen_type].append(gen_val / cap_val)
                else:
                    master_accreditation[gen_type].append(0)

    # 4. Calculate grand average
    final_averages = {}
    for gen_type, percentages in master_accreditation.items():
        if percentages:
            avg_pct = (sum(percentages) / len(percentages)) * 100
            final_averages[gen_type] = avg_pct
            
    # 5. Build and print the DataFrame
    results_list = []
    sorted_averages = sorted(final_averages.items(), key=lambda x: x[1], reverse=True)
    
    for gen_type, pct in sorted_averages:
        clean_name = name_dictionary.get(gen_type, gen_type)
        if clean_name != "Total":
            results_list.append({
                "Generator Type": clean_name,
                "Accreditation (%)": round(pct, 2)
            })
            
    results_df = pd.DataFrame(results_list)
    
    print("\n--- Final Grand Average Accreditation Percentage (>1GW NSE Hours) ---")
    if results_df.empty:
        print("No hours found with NSE > 1 GW across processed scenarios.")
    else:
        print(results_df.to_string(index=False))

Found 400 scenario folders. Processing...

--- Final Grand Average Accreditation Percentage (>1GW NSE Hours) ---
        Generator Type  Accreditation (%)
               Nuclear              92.63
               Biomass              76.19
                  Coal              73.42
     Combined Cycle NG              73.40
 Combustion Turbine NG              66.33
      Steam Turbine NG              58.92
          Pumped Hydro              57.11
             Petroleum              54.14
         Offshore Wind              46.41
         Hydroelectric              38.26
          Onshore Wind              34.56
       Battery Storage              17.98
                 Solar              15.44
Distributed Generation              12.45


In [2]:
import pandas as pd
import glob
import os

# Mapping dictionary for clean display names
name_dictionary = {
    "hydroelectric_pumped_storage": "Pumped Hydro", 
    "nuclear": "Nuclear", 
    "natural_gas_fired_combined_cycle": "Combined Cycle NG", 
    "biomass": "Biomass", 
    "natural_gas_fired_combustion_turbine": "Combustion Turbine NG", 
    "conventional_steam_coal": "Coal", 
    "batteries": "Battery Storage", 
    "conventional_hydroelectric": "Hydroelectric", 
    "solar_photovoltaic": "Solar", 
    "natural_gas_steam_turbine": "Steam Turbine NG", 
    "offshore_wind_turbine": "Offshore Wind", 
    "distributed_generation": "Distributed Generation",
    "onshore_wind_turbine": "Onshore Wind", 
    "petroleum_liquids": "Petroleum"
}

# Function to extract generator type from column name 
def get_generator_type(col_name):
    first_idx = col_name.find('_')
    last_idx = col_name.rfind('_')
    if first_idx != -1 and last_idx != -1 and first_idx != last_idx:
        return col_name[first_idx+1:last_idx]
    return col_name

# Use glob to find all results folders
folder_pattern = "Scenarios/Run_Variability_*_*_R*/results"
result_folders = glob.glob(folder_pattern)

if not result_folders:
    print("No folders found. Please check your working directory and folder path.")
else:
    print(f"Found {len(result_folders)} scenario folders. Processing...")

    summer_accreditation = {}
    winter_accreditation = {}

    for folder in result_folders:
        folder_lower = folder.lower()
        if "summer" in folder_lower:
            season_dict = summer_accreditation
        elif "winter" in folder_lower:
            season_dict = winter_accreditation
        else:
            continue

        nse_path = os.path.join(folder, "nse.csv")
        power_path = os.path.join(folder, "power.csv")
        cap_path = os.path.join(folder, "capacity.csv")
        
        if not (os.path.exists(nse_path) and os.path.exists(power_path) and os.path.exists(cap_path)):
            continue
            
        # 1. Read NSE and identify hours > 1 GW (1000 MW)
        nse_df = pd.read_csv(nse_path, index_col=0)
        nse_df = nse_df.drop(['Zone', 'AnnualSum'], errors='ignore')
        
        # Calculate total hourly NSE across all zones
        hourly_nse = nse_df.apply(pd.to_numeric, errors='coerce').sum(axis=1)
        
        # NEW METHODOLOGY: Filter for all hours where total NSE > 1000
        relevant_hours = hourly_nse[hourly_nse > 100].index.tolist()
        
        if not relevant_hours:
            continue
        
        # 2. Read Capacity
        cap_df = pd.read_csv(cap_path)
        cap_df['GenType'] = cap_df['Resource'].apply(get_generator_type)
        total_cap_by_type = cap_df.groupby('GenType')['EndCap'].sum().to_dict()
        
        for gen_type in total_cap_by_type.keys():
            if gen_type not in season_dict:
                season_dict[gen_type] = []
                
        # 3. Read Power and calculate performance
        power_df = pd.read_csv(power_path, index_col=0)
        
        for hr in relevant_hours:
            if hr not in power_df.index:
                continue
            
            row = power_df.loc[hr].drop('Total', errors='ignore')
            
            type_sums = {}
            for col_name, val in row.items():
                gen_type = get_generator_type(col_name)
                type_sums[gen_type] = type_sums.get(gen_type, 0) + float(val)
                
            for gen_type in total_cap_by_type.keys():
                gen_val = type_sums.get(gen_type, 0)
                cap_val = total_cap_by_type.get(gen_type, 0)
                
                if cap_val > 0:
                    season_dict[gen_type].append(gen_val / cap_val)
                else:
                    season_dict[gen_type].append(0)

    # Helper function to compute the averages from the dictionaries
    def compute_averages(accreditation_dict):
        final_averages = {}
        for gen_type, percentages in accreditation_dict.items():
            if percentages:
                final_averages[gen_type] = (sum(percentages) / len(percentages)) * 100
        return final_averages

    # Calculate final averages for both seasons
    summer_avg = compute_averages(summer_accreditation)
    winter_avg = compute_averages(winter_accreditation)

    # Get a unique list of all generator types found across both seasons
    all_gen_types = set(summer_avg.keys()).union(set(winter_avg.keys()))

    # Build the list of rows for the DataFrame
    results_list = []
    for gen_type in all_gen_types:
        clean_name = name_dictionary.get(gen_type, gen_type)
        
        if clean_name != "Total":
            s_val = summer_avg.get(gen_type, 0.0)
            w_val = winter_avg.get(gen_type, 0.0)
            
            results_list.append({
                "Generator Type": clean_name,
                "Summer Accreditation (%)": round(s_val, 2),
                "Winter Accreditation (%)": round(w_val, 2)
            })

    # Convert to DataFrame
    seasonal_results_df = pd.DataFrame(results_list)

    if seasonal_results_df.empty:
        print("\nNo hours found with NSE > 1 GW across any processed scenarios.")
    else:
        # Sort by Summer Accreditation descending
        seasonal_results_df = seasonal_results_df.sort_values(by="Summer Accreditation (%)", ascending=False).reset_index(drop=True)

        print("\n--- Final Seasonal Average Accreditation Percentage (>1GW NSE Hours) ---")
        print(seasonal_results_df.to_string(index=False))

Found 400 scenario folders. Processing...

--- Final Seasonal Average Accreditation Percentage (>1GW NSE Hours) ---
        Generator Type  Summer Accreditation (%)  Winter Accreditation (%)
          Pumped Hydro                     99.71                     39.59
               Nuclear                     90.78                     93.39
     Combined Cycle NG                     89.58                     66.75
               Biomass                     84.83                     72.64
 Combustion Turbine NG                     75.83                     62.42
                  Coal                     74.20                     73.09
       Battery Storage                     39.75                      9.03
         Hydroelectric                     35.24                     39.50
                 Solar                     31.16                      8.98
         Offshore Wind                     27.82                     54.05
Distributed Generation                     25.20           

In [3]:
results_df.to_csv("Accreditation/annual_accreditation_values.csv")

seasonal_results_df.to_csv("Accreditation/seasonal_accreditation_values.csv")

print("Exported")

Exported
